# Normalization Processing Job

Launches `src/normalize.py` as a SageMaker Processing Job.
Reads the 6 raw source TSVs from S3 and writes normalized versions to `data/normalized/`.

**Run this once.** Output files:
```
data/normalized/train/norm_s1.tsv
data/normalized/train/norm_s2.tsv
data/normalized/train/norm_s3.tsv
data/normalized/test/norm_test_s1.tsv
data/normalized/test/norm_test_s2.tsv
data/normalized/test/norm_test_s3.tsv
```

In [3]:
import boto3
import sagemaker
from sagemaker.sklearn import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

boto_session = boto3.Session(region_name='ap-southeast-2')
session      = sagemaker.Session(boto_session=boto_session)
bucket       = session.default_bucket()
role         = sagemaker.get_execution_role(sagemaker_session=session)  # ← pass session here
PREFIX       = 'entity-resolution-challenge'

print(f'Bucket : s3://{bucket}/{PREFIX}/')
print(f'Role   : {role[:60]}...')

Bucket : s3://sagemaker-ap-southeast-2-725335003020/entity-resolution-challenge/
Role   : arn:aws:iam::725335003020:role/service-role/AmazonSageMaker-...


In [19]:
from sagemaker import image_uris
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

image_uri = image_uris.retrieve(
    framework='sklearn',
    region=boto_session.region_name,
    version='1.2-1',
    image_scope='training',
)

processor = ScriptProcessor(
    image_uri=image_uri,
    command=['python3'],
    instance_type='ml.t3.xlarge',
    instance_count=1,
    role=role,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


In [10]:
import os

# find where the notebook is running from
print(os.getcwd())

/home/ec2-user/SageMaker/Amazon_ML_GMNR/notebooks


In [20]:
processor.run(
    code='/home/ec2-user/SageMaker/Amazon_ML_GMNR/src/normalize.py',
    inputs=[
        ProcessingInput(
            source=f's3://{bucket}/{PREFIX}/data/',
            destination='/opt/ml/processing/input',
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{bucket}/{PREFIX}/data/normalized/',
        )
    ],
)

print(f'\nDone. Output at: s3://{bucket}/{PREFIX}/data/normalized/')

INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2026-09-26-01-15-58-523


....................[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
train/train_source1.tsv → train/norm_s1.tsv
train/train_source2.tsv → train/norm_s2.tsv
train/train_source3.tsv → train/norm_s3.tsv
test/test_source1.tsv → test/norm_test_s1.tsv
test/test_source2.tsv → test/norm_test_s2.tsv
test/test_source3.tsv → test/norm_test_s3.tsv


Done. Output at: s3://sagemaker-ap-southeast-2-725335003020/entity-resolution-challenge/data/normalized/


## Verify output

Check that all 6 files were written and spot-check a few rows.

In [21]:
import boto3
import pandas as pd

s3 = boto3.client('s3')

expected_keys = [
    f'{PREFIX}/data/normalized/train/norm_s1.tsv',
    f'{PREFIX}/data/normalized/train/norm_s2.tsv',
    f'{PREFIX}/data/normalized/train/norm_s3.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s1.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s2.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s3.tsv',
]

for key in expected_keys:
    try:
        obj  = s3.head_object(Bucket=bucket, Key=key)
        size = obj['ContentLength'] // 1024
        print(f'  ✓  {key.split("/")[-1]:25s}  {size:>6} KB')
    except Exception:
        print(f'  ✗  MISSING: {key}')

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


  ✓  norm_s1.tsv                393029 KB
  ✓  norm_s2.tsv                880085 KB
  ✓  norm_s3.tsv                929917 KB
  ✓  norm_test_s1.tsv           322644 KB
  ✓  norm_test_s2.tsv           897758 KB
  ✓  norm_test_s3.tsv           917620 KB


In [22]:
# Spot-check norm_s1
norm_s1 = pd.read_csv(
    f's3://{bucket}/{PREFIX}/data/normalized/train/norm_s1.tsv',
    sep='\t', dtype=str,
)

print(f'norm_s1: {len(norm_s1):,} rows, {len(norm_s1.columns)} columns')
print(f'Columns: {list(norm_s1.columns)}\n')
norm_s1.head(5)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


norm_s1: 2,206,821 rows, 11 columns
Columns: ['entity_id', 'country', 'name_norm', 'name_core', 'name_tokens', 'legal_suffix', 'addr_norm', 'addr_tokens', 'pin_zip', 'has_pin', 'landmark']



,entity_id,country,name_norm,name_core,name_tokens,legal_suffix,addr_norm,addr_tokens,pin_zip,has_pin,landmark
0,S1-925783039,US,orelee s barbershop,orelee s barbershop,barbershop orelee s,NaN,1795 westchester dr high point nc,1795 dr high nc point westchester,NaN,False,NaN
1,S1-773889195,US,prime money,prime money,money prime,NaN,17560 ellis rd tahlequah ok,17560 ellis ok rd tahlequah,17560,True,NaN
2,S1-377745466,US,b retail inc,b retail,b inc retail,inc,1712 montebello ave phoenix az,1712 ave az montebello phoenix,NaN,False,NaN
3,S1-133037285,US,christ chapel,christ chapel,chapel christ,NaN,2100 cameron dr unit apt g dundalk md,2100 apt cameron dr dundalk g md unit,NaN,False,NaN
4,S1-755362802,India,prabhav business center,prabhav business,business center prabhav,center,797 lake town blk a kolkata howrah w bengal,797 a bengal blk howrah kolkata lake town w,NaN,False,NaN


In [28]:
import boto3

s3 = boto3.client("s3")

bucket = "sagemaker-ap-southeast-2-725335003020"
prefix = "entity-resolution-challenge/data/normalized/"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=prefix
)

files = response.get("Contents", [])

if files:
    for obj in files:
        print(obj["Key"], f'{obj["Size"] / (1024**2):.2f} MB')
else:
    print("No files found under this S3 prefix.")

entity-resolution-challenge/data/normalized/test/norm_test_s1.tsv 315.08 MB
entity-resolution-challenge/data/normalized/test/norm_test_s2.tsv 876.72 MB
entity-resolution-challenge/data/normalized/test/norm_test_s3.tsv 896.11 MB
entity-resolution-challenge/data/normalized/train/norm_s1.tsv 383.82 MB
entity-resolution-challenge/data/normalized/train/norm_s2.tsv 859.46 MB
entity-resolution-challenge/data/normalized/train/norm_s3.tsv 908.12 MB


In [29]:
import os

DEST = "/home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized"
os.makedirs(DEST, exist_ok=True)

for obj in files:
    key = obj["Key"]

    # Skip folder markers
    if key.endswith("/"):
        continue

    filename = os.path.basename(key)
    local_path = os.path.join(DEST, filename)

    s3.download_file(bucket, key, local_path)
    print("Downloaded:", local_path)

print("Files downloaded:", len([
    obj for obj in files if not obj["Key"].endswith("/")
]))

Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_test_s1.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_test_s2.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_test_s3.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s1.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s2.tsv
Downloaded: /home/ec2-user/SageMaker/Amazon_ML_GMNR/data/normalized/norm_s3.tsv
Files downloaded: 6
